# Prototipo de la métrica Macro AP-rIoU

Este notebook es el espacio de experimentación de la issue #9. Saúl y Dolly lo usarán para comprender y comprobar manualmente cada parte de la métrica antes de trasladarla a `src/evaluation/metric.py`.

**Estado actual:** primer experimento de representación de una OBB. Todavía no se calcula rIoU, matching ni AP.

## 1. Contrato que debemos respetar

Una OBB se representa como `(cx, cy, width, height, angle_deg)`. El centro `(cx, cy)`, el ancho y el alto se expresan en píxeles; el ángulo se expresa en grados.

Una predicción completa usa `(frame_id, score, cx, cy, width, height, angle_deg)`. El ground truth no contiene `score` porque es la respuesta correcta y no una estimación del modelo.

En este primer experimento la entrada es una OBB paramétrica y la salida esperada son sus cuatro vértices en píxeles.

In [ ]:
import math

import cv2
import numpy as np
from IPython.display import Image, display

## 2. De parámetros a cuatro vértices

Primero colocamos cuatro esquinas alrededor del origen: `(-w/2, -h/2)`, `(w/2, -h/2)`, `(w/2, h/2)` y `(-w/2, h/2)`. Después las rotamos por `angle_deg` y finalmente trasladamos todas al centro `(cx, cy)`.

La función siguiente es deliberadamente experimental. Cuando entendamos y validemos todos sus casos, la implementación definitiva se escribirá y probará en `metric.py`.

In [ ]:
def experimental_obb_to_vertices(obb):
    cx, cy, width, height, angle_deg = obb
    theta = math.radians(angle_deg)
    rotation = np.array(
        [
            [math.cos(theta), -math.sin(theta)],
            [math.sin(theta), math.cos(theta)],
        ],
        dtype=np.float64,
    )
    local_vertices = np.array(
        [
            [-width / 2, -height / 2],
            [width / 2, -height / 2],
            [width / 2, height / 2],
            [-width / 2, height / 2],
        ],
        dtype=np.float64,
    )
    return local_vertices @ rotation.T + np.array([cx, cy])


axis_aligned_obb = (200.0, 150.0, 120.0, 60.0, 0.0)
axis_aligned_vertices = experimental_obb_to_vertices(axis_aligned_obb)
expected_vertices = np.array(
    [[140.0, 120.0], [260.0, 120.0], [260.0, 180.0], [140.0, 180.0]]
)

assert np.allclose(axis_aligned_vertices, expected_vertices)
assert math.isclose(cv2.contourArea(axis_aligned_vertices.astype(np.float32)), 120.0 * 60.0)

print("Vértices calculados:")
print(axis_aligned_vertices)
print()
print("Comprobación: área del polígono = width × height = 7200 px²")

## 3. Visualización de una OBB rotada

Ahora conservamos el mismo centro, ancho y alto, pero usamos un ángulo de 30°. La rotación debe cambiar los vértices sin cambiar el centro ni el área.

In [ ]:
rotated_obb = (200.0, 150.0, 120.0, 60.0, 30.0)
rotated_vertices = experimental_obb_to_vertices(rotated_obb)
rotated_area = cv2.contourArea(rotated_vertices.astype(np.float32))

assert math.isclose(rotated_area, 120.0 * 60.0, rel_tol=1e-6)
assert np.allclose(rotated_vertices.mean(axis=0), [200.0, 150.0])

canvas = np.full((300, 400, 3), 245, dtype=np.uint8)
polygon = np.rint(rotated_vertices).astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(canvas, [polygon], isClosed=True, color=(40, 120, 220), thickness=3)
cv2.circle(canvas, (200, 150), radius=5, color=(220, 60, 40), thickness=-1)
cv2.putText(
    canvas,
    "centro (200, 150)",
    (210, 145),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (40, 40, 40),
    1,
    cv2.LINE_AA,
)
success, encoded_image = cv2.imencode(".png", canvas)
assert success

print(f"Centro recuperado: {rotated_vertices.mean(axis=0)}")
print(f"Área después de rotar: {rotated_area:.1f} px²")
display(Image(data=encoded_image.tobytes()))

## 4. Qué debemos entender antes de continuar

1. `cx` y `cy` indican el centro, no una esquina.
2. Antes de rotar, las esquinas se construyen usando la mitad del ancho y del alto.
3. La rotación cambia la posición de los vértices, pero conserva el centro y el área.
4. Los vértices son necesarios porque rIoU compara la intersección de dos polígonos rotados.

**Siguiente experimento:** validar dimensiones y ángulos problemáticos antes de calcular la intersección entre dos OBB.

## 5. Validación experimental de una OBB

No todas las tuplas de cinco números describen una caja geométrica válida. El ancho y el alto deben ser estrictamente positivos; además, el centro, las dimensiones y el ángulo deben ser valores finitos. NaN e infinito harían imposible obtener vértices confiables.

In [ ]:
def experimental_validate_obb(obb):
    values = np.asarray(obb, dtype=np.float64)
    if values.shape != (5,):
        raise ValueError("Una OBB debe contener exactamente cinco valores")
    if not np.all(np.isfinite(values)):
        raise ValueError("Todos los valores de la OBB deben ser finitos")
    if values[2] <= 0 or values[3] <= 0:
        raise ValueError("El ancho y el alto deben ser mayores que cero")
    return tuple(float(value) for value in values)


valid_obb = (200.0, 150.0, 120.0, 60.0, 30.0)
assert experimental_validate_obb(valid_obb) == valid_obb

invalid_obbs = {
    "ancho cero": (200.0, 150.0, 0.0, 60.0, 30.0),
    "altura negativa": (200.0, 150.0, 120.0, -60.0, 30.0),
    "centro NaN": (math.nan, 150.0, 120.0, 60.0, 30.0),
    "ángulo infinito": (200.0, 150.0, 120.0, 60.0, math.inf),
}

for case_name, invalid_obb in invalid_obbs.items():
    try:
        experimental_validate_obb(invalid_obb)
    except ValueError as error:
        print(f"✓ {case_name}: rechazada ({error})")
    else:
        raise AssertionError(f"La OBB inválida '{case_name}' fue aceptada")

## 6. Normalización y equivalencia de ángulos

Una vuelta completa tiene 360°. Por eso, sumar o restar 360° no cambia la orientación de la caja. Normalizar con módulo 360 lleva cualquier ángulo finito al intervalo [0°, 360°). Por ejemplo, -15° se convierte en 345°.

In [ ]:
def experimental_normalize_angle(angle_deg):
    if not math.isfinite(angle_deg):
        raise ValueError("El ángulo debe ser finito")
    return angle_deg % 360.0


negative_angle = -15.0
normalized_angle = experimental_normalize_angle(negative_angle)
assert normalized_angle == 345.0

negative_angle_obb = (200.0, 150.0, 120.0, 60.0, negative_angle)
normalized_angle_obb = (200.0, 150.0, 120.0, 60.0, normalized_angle)
negative_vertices = experimental_obb_to_vertices(negative_angle_obb)
normalized_vertices = experimental_obb_to_vertices(normalized_angle_obb)

assert np.allclose(negative_vertices, normalized_vertices, atol=1e-9)

print(f"{negative_angle}° normalizado = {normalized_angle}°")
print(
    "Diferencia máxima entre sus vértices:",
    float(np.max(np.abs(negative_vertices - normalized_vertices))),
)

## 7. Conclusiones de este experimento

1. Una dimensión igual a cero no forma una superficie y debe rechazarse.
2. Una dimensión negativa no tiene interpretación geométrica y debe rechazarse.
3. NaN e infinito contaminarían todos los cálculos posteriores.
4. -15° y 345° representan la misma orientación y producen los mismos vértices.
5. Estas reglas se trasladarán después a validate_obb() y normalize_angle() dentro de metric.py.

**Siguiente experimento:** dibujar dos OBB superpuestas y comprender visualmente intersección, unión y rIoU.

## 8. Intersección, unión y rIoU

La intersección es el área cubierta simultáneamente por las dos OBB. La unión es toda el área cubierta por al menos una de ellas y se calcula como área A + área B - intersección. Finalmente, rIoU = intersección / unión.

Usaremos primero un ejemplo comprobable a mano. Cada caja mide 120 × 80, por lo que su área es 9600 px². Al desplazar la segunda caja 40 px horizontalmente, comparten una región de 80 × 80 = 6400 px². La unión es 9600 + 9600 - 6400 = 12800 px² y rIoU debe ser 6400 / 12800 = 0.5.

In [ ]:
def experimental_rotated_iou(obb_a, obb_b):
    valid_a = experimental_validate_obb(obb_a)
    valid_b = experimental_validate_obb(obb_b)
    polygon_a = experimental_obb_to_vertices(valid_a).astype(np.float32)
    polygon_b = experimental_obb_to_vertices(valid_b).astype(np.float32)

    area_a = float(cv2.contourArea(polygon_a))
    area_b = float(cv2.contourArea(polygon_b))
    intersection_area, intersection_polygon = cv2.intersectConvexConvex(
        polygon_a, polygon_b
    )
    intersection_area = float(intersection_area)
    union_area = area_a + area_b - intersection_area
    if union_area <= 0.0:
        return 0.0, intersection_area, union_area, intersection_polygon

    riou = min(1.0, max(0.0, intersection_area / union_area))
    return riou, intersection_area, union_area, intersection_polygon


box_a = (160.0, 150.0, 120.0, 80.0, 0.0)
box_b = (200.0, 150.0, 120.0, 80.0, 0.0)
riou, intersection_area, union_area, intersection_polygon = (
    experimental_rotated_iou(box_a, box_b)
)

assert math.isclose(intersection_area, 6400.0)
assert math.isclose(union_area, 12800.0)
assert math.isclose(riou, 0.5)

print(f"Intersección: {intersection_area:.1f} px²")
print(f"Unión: {union_area:.1f} px²")
print(f"rIoU: {riou:.2f}")

In [ ]:
canvas = np.full((300, 400, 3), 245, dtype=np.uint8)
vertices_a = experimental_obb_to_vertices(box_a)
vertices_b = experimental_obb_to_vertices(box_b)
polygon_a_draw = np.rint(vertices_a).astype(np.int32).reshape((-1, 1, 2))
polygon_b_draw = np.rint(vertices_b).astype(np.int32).reshape((-1, 1, 2))

if intersection_polygon is not None and len(intersection_polygon) >= 3:
    intersection_draw = np.rint(intersection_polygon).astype(np.int32)
    overlay = canvas.copy()
    cv2.fillPoly(overlay, [intersection_draw], color=(90, 200, 90))
    canvas = cv2.addWeighted(overlay, 0.55, canvas, 0.45, 0.0)

cv2.polylines(canvas, [polygon_a_draw], True, (220, 80, 60), 3)
cv2.polylines(canvas, [polygon_b_draw], True, (60, 100, 220), 3)
cv2.circle(canvas, (160, 150), 4, (220, 80, 60), -1)
cv2.circle(canvas, (200, 150), 4, (60, 100, 220), -1)
cv2.putText(canvas, "A", (145, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220, 80, 60), 2)
cv2.putText(canvas, "B", (215, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (60, 100, 220), 2)

success, encoded_intersection = cv2.imencode(".png", canvas)
assert success
display(Image(data=encoded_intersection.tobytes()))

## 9. Casos extremos de rIoU

Una implementación correcta debe devolver 1 para cajas idénticas, 0 para cajas separadas y el mismo resultado sin importar el orden de las cajas. En una caja no cuadrada, cambiar únicamente su ángulo debe reducir el solapamiento.

In [ ]:
identical_riou = experimental_rotated_iou(box_a, box_a)[0]
separated_box = (340.0, 150.0, 120.0, 80.0, 0.0)
separated_riou = experimental_rotated_iou(box_a, separated_box)[0]
reverse_riou = experimental_rotated_iou(box_b, box_a)[0]
rotated_box = (160.0, 150.0, 120.0, 80.0, 30.0)
angular_deviation_riou = experimental_rotated_iou(box_a, rotated_box)[0]
equivalent_angle_riou = experimental_rotated_iou(
    negative_angle_obb, normalized_angle_obb
)[0]

assert math.isclose(identical_riou, 1.0)
assert math.isclose(separated_riou, 0.0)
assert math.isclose(reverse_riou, riou)
assert 0.0 < angular_deviation_riou < 1.0
assert math.isclose(equivalent_angle_riou, 1.0, rel_tol=1e-6)

print(f"Cajas idénticas: {identical_riou:.2f}")
print(f"Cajas separadas: {separated_riou:.2f}")
print(f"Orden inverso: {reverse_riou:.2f}")
print(f"Desviación angular de 30°: {angular_deviation_riou:.4f}")
print(f"Ángulos -15° y 345°: {equivalent_angle_riou:.2f}")

## 10. Conclusiones del experimento de rIoU

1. rIoU mide cuánto se superponen dos OBB en relación con toda el área que cubren.
2. Restar la intersección al sumar ambas áreas evita contar dos veces la zona compartida.
3. El resultado siempre debe permanecer entre 0 y 1.
4. Rotar una caja no cuadrada puede reducir rIoU aunque conserve el mismo centro y área.
5. La implementación definitiva se migrará posteriormente a rotated_iou() dentro de metric.py.

**Siguiente experimento pendiente:** simular el matching uno a uno entre predicciones ordenadas por score y ground truths.